# Tabular Parsing

In [5]:
from pathlib import Path

from src.lake_agent.indexing.tabular import (
    DeterministicTabularParser,
    TabularLLMEnricher,
)

DATA_DIR = Path("./data/Data-Lake")
SUPPORTED_EXTENSIONS = {".xlsx", ".csv", ".tsv"}

parser = DeterministicTabularParser()
enricher = TabularLLMEnricher.from_env()

files = sorted(
    path
    for path in DATA_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

print(f"Found {len(files)} tabular files")

enriched_results = []
failed_files = []

for path in files:
    print(f"Processing: {path}")

    try:
        result = parser.parse_file(str(path))
        enriched_result = enricher.enrich(result)
        enriched_results.append(enriched_result)

    except Exception as exc:
        failed_files.append((path, exc))
        print(f"Failed: {path}")
        print(f"Reason: {exc}")

print(f"Enriched results: {len(enriched_results)}")
print(f"Failed files: {len(failed_files)}")

Found 22 tabular files
Processing: data/Data-Lake/archeology/climateMeasurements.xlsx
Processing: data/Data-Lake/archeology/conflict_brecke.csv
Processing: data/Data-Lake/archeology/radiocarbon_database_regional.xlsx
Processing: data/Data-Lake/archeology/roman_cities.csv
Processing: data/Data-Lake/biomedical/1-s2.0-S0092867420301070-mmc1.xlsx
Processing: data/Data-Lake/biomedical/1-s2.0-S0092867420301070-mmc2.xlsx
Processing: data/Data-Lake/biomedical/1-s2.0-S0092867420301070-mmc3.xlsx
Processing: data/Data-Lake/biomedical/1-s2.0-S0092867420301070-mmc4.xlsx
Processing: data/Data-Lake/biomedical/1-s2.0-S0092867420301070-mmc5.xlsx
Processing: data/Data-Lake/biomedical/1-s2.0-S0092867420301070-mmc6.xlsx
Processing: data/Data-Lake/biomedical/1-s2.0-S0092867420301070-mmc7.xlsx
Processing: data/Data-Lake/biomedical/hyperactivated.csv
Processing: data/Data-Lake/da-dev-tables/Credit.csv
Processing: data/Data-Lake/da-dev-tables/Current_Logan.csv
Processing: data/Data-Lake/da-dev-tables/bitconne

In [8]:
from src.lake_agent.indexing.tabular.vector_store import (
    build_pgvector_store,
    add_tabular_result,
)

vector_store = build_pgvector_store(
    collection_name="lakeagent_tabular_index",
)

indexed_ids_by_file = {}
failed_indexing = []

for enriched_result in enriched_results:
    try:
        indexed_ids = add_tabular_result(
            vector_store=vector_store,
            result=enriched_result,
        )

        indexed_ids_by_file[enriched_result.relative_path] = indexed_ids

        print(f"Indexed {len(indexed_ids)} records from {enriched_result.relative_path}")
        print(indexed_ids)
        print("-" * 80)

    except Exception as exc:
        failed_indexing.append(
            {
                "relative_path": enriched_result.relative_path,
                "filename": enriched_result.filename,
                "error": repr(exc),
            }
        )

        print(f"Failed indexing: {enriched_result.relative_path}")
        print(f"Reason: {exc}")
        print("-" * 80)

print("=" * 80)
print(f"Successfully indexed files: {len(indexed_ids_by_file)}")
print(f"Failed indexing files: {len(failed_indexing)}")
print(f"Total indexed records: {sum(len(ids) for ids in indexed_ids_by_file.values())}")

Indexed 2 records from climateMeasurements.xlsx
['source_5f2e520e3500cd12:file', 'table_f00600be5465bb2f']
--------------------------------------------------------------------------------
Indexed 2 records from conflict_brecke.csv
['source_cfabdbb6673f4d2e:file', 'table_0c9aadad99d0636d']
--------------------------------------------------------------------------------
Indexed 2 records from radiocarbon_database_regional.xlsx
['source_26b404e86c1d009b:file', 'table_95984a9154198563']
--------------------------------------------------------------------------------
Indexed 2 records from roman_cities.csv
['source_51e3e9d3a425859d:file', 'table_f8155217a3a58f8c']
--------------------------------------------------------------------------------
Indexed 2 records from 1-s2.0-S0092867420301070-mmc1.xlsx
['source_4b78948b8ea857d4:file', 'table_af4f90e13177440b']
--------------------------------------------------------------------------------
Indexed 3 records from 1-s2.0-S0092867420301070-mmc2.

In [25]:
query = (
    '''
    Determine the correlation coefficient between the "Limit" and "Balance" 
    columns in the Credit.csv file where "correlation_value" is the calculated
    Pearson correlation coefficient between "Limit" and "Balance", rounded to two decimal places.
    '''
)

results = vector_store.similarity_search_with_score(
    query,
    k=5,
)

for doc, score in results:
    print("score:", score)
    print("record_type:", doc.metadata.get("record_type"))
    print("filename:", doc.metadata.get("filename"))
    print("table_id:", doc.metadata.get("table_id"))
    print("table_name:", doc.metadata.get("table_name"))
    print("sheet_name:", doc.metadata.get("sheet_name"))
    print()
    print(doc.page_content[:800])
    print("-" * 80)

score: 0.6641199027378398
record_type: table
filename: Credit.csv
table_id: table_1e026734ff2c5eec
table_name: Credit
sheet_name: None

Income | Limit | Rating | Cards | Age | Education | Gender | Student | Married | Ethnicity | Balance
Credit card account holders with financial metrics (income, credit limit, rating, balance) and demographic attributes (age, education, gender, student status, marital status, ethnicity, number of cards)
credit limit, income, credit rating, balance, age, education, gender, student
--------------------------------------------------------------------------------
score: 0.6779715162086397
record_type: file
filename: Credit.csv
table_id: None
table_name: None
sheet_name: None

Credit card holder demographic and account dataset with income, credit limit, rating, and balance information. Contains individual-level records with personal attributes including age, education, gender, student status, marital status, and ethnicity.

credit card, income, credit limit,